<a href="https://colab.research.google.com/github/PriyanshuBhunia/classification-ML/blob/main/CIFAR100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import f1_score
import numpy as np

# 1. Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 2. Hyperparameters
batch_size = 64
learning_rate = 0.001
num_epochs = 5

# 3. Data Preparation
# CIFAR-100 Mean and Std
stats = ((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

# Load datasets
full_train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

# Split training into train (40k) and validation (10k)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# 4. Model Definitions

# MLP Model
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 100)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

# CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Linear(512, 100)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 5. Training and Evaluation Functions

def train_model(model, train_loader, val_loader, num_epochs, learning_rate, model_name="Model"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Starting training for {model_name}...")
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Val Acc: {val_acc:.2f}%")

    return model

def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return accuracy, f1

# 6. Execution

# Train and Evaluate MLP
print("\n--- MLP --- ")
mlp = MLP()
mlp = train_model(mlp, train_loader, val_loader, num_epochs, learning_rate, "MLP")
mlp_acc, mlp_f1 = evaluate_model(mlp, test_loader)
print(f"MLP Test Accuracy: {mlp_acc:.2f}%")
print(f"MLP F1 Score (Macro): {mlp_f1:.4f}")

# Train and Evaluate CNN
print("\n--- CNN --- ")
cnn = CNN()
cnn = train_model(cnn, train_loader, val_loader, num_epochs, learning_rate, "CNN")
cnn_acc, cnn_f1 = evaluate_model(cnn, test_loader)
print(f"CNN Test Accuracy: {cnn_acc:.2f}%")
print(f"CNN F1 Score (Macro): {cnn_f1:.4f}")

Using device: cpu

--- MLP --- 
Starting training for MLP...
Epoch [1/5], Loss: 3.9981, Val Acc: 12.11%
Epoch [2/5], Loss: 3.7079, Val Acc: 13.35%
Epoch [3/5], Loss: 3.5947, Val Acc: 15.47%
Epoch [4/5], Loss: 3.5231, Val Acc: 16.75%
Epoch [5/5], Loss: 3.4690, Val Acc: 16.85%
MLP Test Accuracy: 13.04%
MLP F1 Score (Macro): 0.1106

--- CNN --- 
Starting training for CNN...
Epoch [1/5], Loss: 3.8074, Val Acc: 18.98%
Epoch [2/5], Loss: 3.1718, Val Acc: 25.29%
Epoch [3/5], Loss: 2.8719, Val Acc: 29.23%
Epoch [4/5], Loss: 2.6499, Val Acc: 32.91%
Epoch [5/5], Loss: 2.4814, Val Acc: 34.50%
CNN Test Accuracy: 37.79%
CNN F1 Score (Macro): 0.3628
